In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

In [3]:
# ====================================================================
# KONFIGURASI
# Sesuaikan path sesuai lokasi file di Google Drive kamu.
# ====================================================================

BASE = '/content/drive/MyDrive/Colab Datasets/Augmented Data'
BASED = '/content/drive/MyDrive/Colab Datasets'

# --- Kolom & tipe data wajib yang harus ada di setiap batch ---
SCHEMA = ['id', 'text', 'label', 'language', 'has_slang', 'has_abbrev', 'source']

DTYPE_CASTS = {
    'id'        : int,
    'text'      : str,
    'label'     : int,
    'has_slang' : int,
    'has_abbrev': int,
    'language'  : str,
    'source'    : str,
}

# --- Path 43 batch ---
BATCH_PATHS = [f'{BASE}/batch{i}_codemixed.csv' for i in range(1, 44)]

# --- Output akhir ---
OUTPUT_PATH = f'{BASED}/combined_augmented_codemixed_dataset.csv'

# --- (Opsional) Path audit CSV; set None untuk skip ---
AUDIT_PATH = None   # contoh: f'{BASE}/audit_merged.csv'

ROWS_PER_BATCH = 400  # jumlah baris yang diharapkan per batch


In [4]:
# ====================================================================
# HELPER: validate_schema
# Memastikan semua kolom wajib ada.  Kolom ekstra di-drop agar
# schema seragam.  Kembalikan df dengan urutan kolom = SCHEMA.
# ====================================================================

def validate_schema(df: pd.DataFrame, schema: list, label: str = '') -> pd.DataFrame:
    missing = [col for col in schema if col not in df.columns]
    if missing:
        raise ValueError(
            f"[ERROR] {label}: kolom berikut tidak ditemukan: {missing}\n"
            f"  Kolom yang ada: {list(df.columns)}"
        )
    extra = [col for col in df.columns if col not in schema]
    if extra:
        print(f"    [WARN] {label}: kolom ekstra di-drop: {extra}")
        df = df.drop(columns=extra)
    return df[schema]


# ====================================================================
# HELPER: cast_dtypes
# Terapkan casting tipe data sesuai DTYPE_CASTS.
# ====================================================================

def cast_dtypes(df: pd.DataFrame, casts: dict, label: str = '') -> pd.DataFrame:
    for col, dtype in casts.items():
        if col not in df.columns:
            continue
        try:
            if dtype == str:
                df[col] = df[col].astype(str).str.strip()
            else:
                df[col] = df[col].astype(dtype)
        except (ValueError, TypeError) as e:
            raise ValueError(f"[ERROR] {label}: gagal cast kolom '{col}' ke {dtype.__name__}: {e}")
    return df


In [5]:
SEP_MAIN = "=" * 64
SEP_SUB  = "-" * 56

print(f"\n{SEP_MAIN}")
print(f"  MERGE: 43 BATCH DATASET CODE-MIXED")
print(f"{SEP_MAIN}")

# ----------------------------------------------------------------
# [1] LOAD 43 BATCH
# ----------------------------------------------------------------
print(f"\n[1] LOAD 43 BATCH")
print(f"    {'Batch':<8} {'Baris':>6}  {'Hate':>6}  {'Non-hate':>9}  {'ID min':>7}  {'ID max':>7}")
print(f"    {'-'*55}")

batch_dfs   = []
warnings    = []

for i, path in enumerate(BATCH_PATHS, start=1):
    try:
        df_batch = pd.read_csv(path)
    except FileNotFoundError:
        raise FileNotFoundError(f"[ERROR] File tidak ditemukan: {path}")

    df_batch = validate_schema(df_batch, SCHEMA, label=f'batch{i}')
    df_batch = cast_dtypes(df_batch, DTYPE_CASTS, label=f'batch{i}')

    # Validasi jumlah baris per batch
    if len(df_batch) != ROWS_PER_BATCH:
        msg = (f"[WARN] batch{i}: jumlah baris {len(df_batch):,} "               f"(expected {ROWS_PER_BATCH:,})")
        warnings.append(msg)

    # Validasi label hanya 0 atau 1
    invalid_labels = df_batch[~df_batch['label'].isin([0, 1])]
    if not invalid_labels.empty:
        raise ValueError(
            f"[ERROR] batch{i}: ditemukan nilai label tidak valid: "            f"{invalid_labels['label'].unique().tolist()}"        )

    # Validasi has_slang dan has_abbrev hanya 0 atau 1
    for flag_col in ['has_slang', 'has_abbrev']:
        invalid_flags = df_batch[~df_batch[flag_col].isin([0, 1])]
        if not invalid_flags.empty:
            raise ValueError(
                f"[ERROR] batch{i}: ditemukan nilai '{flag_col}' tidak valid: "                f"{invalid_flags[flag_col].unique().tolist()}"            )

    n_hate   = (df_batch['label'] == 1).sum()
    n_nohate = (df_batch['label'] == 0).sum()
    id_min   = df_batch['id'].min()
    id_max   = df_batch['id'].max()

    print(f"    batch{i:<3}  {len(df_batch):>6,}  {n_hate:>6,}  {n_nohate:>9,}  "          f"{id_min:>7,}  {id_max:>7,}")
    batch_dfs.append(df_batch)

# Cetak warning setelah tabel
if warnings:
    print()
    for w in warnings:
        print(f"    {w}")

print()
print(f"    {len(batch_dfs)} batch berhasil dimuat.")

# ----------------------------------------------------------------
# [2] CONCAT (tanpa dedup)
# ----------------------------------------------------------------
print(f"\n[2] CONCAT SEMUA BATCH")
df_final = pd.concat(batch_dfs, ignore_index=False).reset_index(drop=True)
print(f"    Total baris setelah concat : {len(df_final):,}")

# ----------------------------------------------------------------
# [3] VERIFIKASI ID SEQUENTIAL & KONTINUITAS
# ----------------------------------------------------------------
print(f"\n[3] VERIFIKASI ID")

# Pastikan tidak ada duplikat ID
dup_ids = df_final[df_final.duplicated(subset='id', keep=False)]
if not dup_ids.empty:
    raise ValueError(
        f"[ERROR] Terdapat {len(dup_ids):,} baris dengan ID duplikat: "        f"{sorted(dup_ids['id'].unique())[:20]} ..."    )

# Pastikan ID berurutan dari 1 hingga N tanpa gap
id_sorted  = df_final['id'].sort_values().reset_index(drop=True)
expected   = pd.Series(range(1, len(df_final) + 1))
gap_mask   = id_sorted != expected
if gap_mask.any():
    bad_ids = id_sorted[gap_mask].tolist()[:20]
    raise ValueError(
        f"[ERROR] ID tidak sequential / ada gap. Contoh ID bermasalah: {bad_ids}"    )

print(f"    ID unik          : OK")
print(f"    ID sequential    : OK (1 – {len(df_final):,})")
print(f"    Tidak ada gap    : OK")

# ----------------------------------------------------------------
# [4] VERIFIKASI SCHEMA FINAL
# ----------------------------------------------------------------
print(f"\n[4] VERIFIKASI SCHEMA FINAL")
assert list(df_final.columns) == SCHEMA,     f"[ERROR] Schema tidak sesuai: {list(df_final.columns)}"
print(f"    Kolom : {list(df_final.columns)}")
print(f"    Dtypes: {dict(df_final.dtypes.astype(str))}")
print(f"    Schema OK")

# ----------------------------------------------------------------
# [5] LAPORAN DISTRIBUSI
# ----------------------------------------------------------------
print(f"\n[5] DISTRIBUSI AKHIR")
print(f"    {SEP_SUB}")

for src in df_final['source'].unique():
    sub  = df_final[df_final['source'] == src]
    n_h  = (sub['label'] == 1).sum()
    n_nh = (sub['label'] == 0).sum()
    n_sl = sub['has_slang'].sum()
    n_ab = sub['has_abbrev'].sum()
    print(
        f"    source='{src}' (n={len(sub):,}): "        f"hate={n_h:,}, non-hate={n_nh:,}, "        f"slang={n_sl:,} ({sub['has_slang'].mean()*100:.1f}%), "        f"abbrev={n_ab:,} ({sub['has_abbrev'].mean()*100:.1f}%)"    )

print(f"    {SEP_SUB}")
total_hate   = (df_final['label'] == 1).sum()
total_nohate = (df_final['label'] == 0).sum()
total_sl     = df_final['has_slang'].sum()
total_ab     = df_final['has_abbrev'].sum()
n11 = ((df_final['has_slang'] == 1) & (df_final['has_abbrev'] == 1)).sum()
n10 = ((df_final['has_slang'] == 1) & (df_final['has_abbrev'] == 0)).sum()
n01 = ((df_final['has_slang'] == 0) & (df_final['has_abbrev'] == 1)).sum()
n00 = ((df_final['has_slang'] == 0) & (df_final['has_abbrev'] == 0)).sum()
print(f"    TOTAL ({len(df_final):,} baris)")
print(f"      hate       : {total_hate:,} ({total_hate/len(df_final)*100:.1f}%)")
print(f"      non-hate   : {total_nohate:,} ({total_nohate/len(df_final)*100:.1f}%)")
print(f"      has_slang  : {total_sl:,} ({df_final['has_slang'].mean()*100:.1f}%)")
print(f"      has_abbrev : {total_ab:,} ({df_final['has_abbrev'].mean()*100:.1f}%)")
print(f"      (slang=1, abbrev=1) = {n11:,}")
print(f"      (slang=1, abbrev=0) = {n10:,}")
print(f"      (slang=0, abbrev=1) = {n01:,}")
print(f"      (slang=0, abbrev=0) = {n00:,}")
print(f"    {SEP_SUB}")

# ----------------------------------------------------------------
# [6] SAVE
# ----------------------------------------------------------------
print(f"\n[6] SAVE")

if AUDIT_PATH is not None:
    df_final.to_csv(AUDIT_PATH, index=False)
    print(f"    [6a] Audit CSV  : {AUDIT_PATH}")

df_final.to_csv(OUTPUT_PATH, index=False)
print(f"    [6b] Final CSV  : {OUTPUT_PATH}")
print(f"         Kolom      : {list(df_final.columns)}")
print(f"         Total baris: {len(df_final):,}")

print(f"\n{SEP_MAIN}")
print(f"  RINGKASAN MERGE")
print(f"{SEP_MAIN}")
print(f"  Jumlah batch       : {len(BATCH_PATHS)}")
print(f"  Total baris final  : {len(df_final):,}")
print(f"  Distribusi label   : hate={total_hate:,} | non-hate={total_nohate:,}")
print(f"  Output             : {OUTPUT_PATH}")
print(f"{SEP_MAIN}")


  MERGE: 43 BATCH DATASET CODE-MIXED

[1] LOAD 43 BATCH
    Batch     Baris    Hate   Non-hate   ID min   ID max
    -------------------------------------------------------
    batch1       400     200        200        1      400
    batch2       400     200        200      401      800
    batch3       400     200        200      801    1,200
    batch4       400     200        200    1,201    1,600
    batch5       400     200        200    1,601    2,000
    batch6       400     200        200    2,001    2,400
    batch7       400     200        200    2,401    2,800
    batch8       400     200        200    2,801    3,200
    batch9       400     200        200    3,201    3,600
    batch10      400     200        200    3,601    4,000
    batch11      400     200        200    4,001    4,400
    batch12      400     200        200    4,401    4,800
    batch13      400     200        200    4,801    5,200
    batch14      400     200        200    5,201    5,600
    batch15   

In [6]:
print("5 baris pertama:")
display(df_final.head())
print("\n5 baris terakhir:")
display(df_final.tail())


5 baris pertama:


,id,text,label,language,has_slang,has_abbrev,source
0,1,"gue literally belum makan dr tadi, starving bg...",0,mixed,0,0,generated
1,2,"they keep acting like they're superior, padaha...",1,mixed,0,0,generated
2,3,ngl gue lowkey baper pas dia tiba2 so sweet gi...,0,mixed,0,0,generated
3,4,"semua perempuan tuh emang manipulative, gak ad...",1,mixed,0,0,generated
4,5,"deadline besok tp gue blm buka materi, i think...",0,mixed,0,0,generated



5 baris terakhir:


,id,text,label,language,has_slang,has_abbrev,source
17195,17196,"Selling these offense books should be a crime,...",1,mixed,0,0,generated
17196,17197,"Stop pretense you care, kamu hanya ingin dilihat!",1,mixed,0,0,generated
17197,17198,"Always playing the victim in every post, gak m...",1,mixed,0,0,generated
17198,17199,"They always rich_person the latest movies, jad...",0,mixed,0,0,generated
17199,17200,You think you're tough? Hanya mulut besar saja!,1,mixed,0,0,generated
